# Repeated-structure pseudospace: try to falsify the identifying principle

Exploratory notebook 15. Independent segmented tubule profiles are destructive 2D observations of different nephron instances, **not** longitudinal observations or tracked cells. Existing notebooks remain canonical and unchanged. No fine spatial ground truth is available. No novelty or superiority is assumed.

## 1 — Problem formulation

We posit $x_{ig}=f_g(z_i)+b_{s(i),g}+\epsilon_{ig}$ with unknown canonical position $z_i\in[0,1]$. A specimen contains many independent profiles, not observations with known longitudinal correspondence. Reproducible molecular programs might help choose an ordering; specimen is the independent replication unit. DPT finds a diffusion direction, principal curves fit a smooth path, and OT relates distributions. Here the candidate criterion is prediction of molecular profiles across independent specimens.

**Identifiability ceiling:** for any increasing bijection $h$, replacing $z$ by $h(z)$ and $f$ by $f\circ h^{-1}$ leaves the data model unchanged. Replicate agreement alone does not identify metric distance, sampling density, or absolute position. Reflection is an additional ambiguity without orientation anchors. At best we recover an order and an explicitly chosen coordinate gauge. Anatomy thirds below are a convenient gauge, **not equal physical lengths**. A shared nuisance program can also reproduce across specimens. Recurrence is an assumption, not an identification theorem.

The minimum falsification experiment compares independently predicted **evaluation genes** under existing DPT and modest local changes. Advancement requires better unseen-gene prediction within anatomy, stability, and prospective specimen mapping; more significant pathways or smoother pictures do not count.

## 2 — Focused literature and novelty check

Search date: 2026-10-01. Parallel Search was attempted but no authenticated backend was available; public web search and primary paper/official documentation pages were used. This is a focused scoping review, not exhaustive proof of absence. Exact query families: latent spatial coordinate / repeated anatomical structures / unordered cross-sections / destructive sampling; functional registration / principal curves with replicates; replicate-aware pseudotime / cross-sample trajectory inference; manifold alignment / domain invariant latent representation; anatomically constrained OT / latent position and cross-replicate prediction. Accessible abstracts, methods and author documentation were inspected; publisher access failures were checked against author versions where available.

| Closest method/source | Existing principle | Difference and novelty risk |
|---|---|---|
| [PFI, Maddu et al., PNAS 2025](https://doi.org/10.1073/pnas.2420621122), [author preprint](https://arxiv.org/html/2410.07501v2) | Probability-flow fitting of time-resolved cross-sectional marginals, requiring biophysical/noise assumptions | No known time or physical flow here; predictive recurrence is a different assumption, not a solution of dynamical identification. |
| [Pheno-GS, 2026 preprint](https://arxiv.org/html/2609.27633v1) | Connectivity-regularized geodesic, unbalanced Sinkhorn distances between sample distributions | Geometry/transport could regularize an axis; distribution distances do not themselves estimate canonical structure position. |
| [PhenoPath, 2018](https://www.nature.com/articles/s41467-018-04696-6) | Joint latent pseudotime and covariate-dependent gene effects | Covariates already enter estimation. Separating specimen variation is not new; nonlinear cross-specimen prediction would need a precise objective distinction. |
| [MATCHER, 2017](https://genomebiology.biomedcentral.com/articles/10.1186/s13059-017-1269-0) | GPLVM pseudotime and master-axis alignment of unpaired assays | Shared unobserved axes without correspondence already exist; alignment and curve projection are standard. |
| [MEFISTO, 2022 / author guide](https://biofam.github.io/MOFA2/MEFISTO.html) | Smooth shared/group-specific factors with alignment of supplied continuous covariates | Replicate/group agreement enters the model. Usually starts with a continuous covariate rather than wholly unordered profiles. |
| [TrAGEDy, 2025](https://github.com/No2Ross/TrAGEDy) | Expression-trajectory alignment with partial/divergent matching | Cross-replicate molecular agreement can alter coordinates already; starts from estimated trajectories rather than our proposed cross-fitted mapping loss. |
| [DeLorean, 2016](https://doi.org/10.1093/bioinformatics/btw372) | GP latent pseudotime with uncertain capture-time prior | Smooth latent position estimation and uncertainty are existing components; anatomical intervals replace time information. |
| [novoSpaRc, 2019](https://www.nature.com/articles/s41586-019-1773-3) | Spatial reconstruction from unpaired expression using structural correspondence and optional markers | Latent spatial reconstruction is already formalized, including kidney/one-dimensional settings; a repeated-section application is not enough for novelty. |
| [moscot, 2025](https://www.nature.com/articles/s41586-024-08453-2) | OT mapping across time, space and molecular assays | Inductive atlas mapping and constrained spatial correspondence are established; avoid equating mapping with a new identifying principle. |

**A:** Spatial reconstruction without cell correspondence is established. This search does not establish that the exact repeated-profile problem is absent from the literature. **B:** Yes: groups/covariates enter latent estimation and trajectory alignment, rather than serving only validation. **C:** Related shared latent models and expression-based alignment optimize agreement; no exact specimen-cross-fitted, held-out-gene objective was verified in this bounded search. That is an unresolved search question, not a novelty claim. **D:** This notebook implements a combination of standard spline regression, cross-fitting, feature ranking and nearest-curve projection. A genuinely distinct objective would require both a deeper precedence audit and evidence beyond DPT-initialized registration.

PFI motivates stating assumptions; it does **not** justify treating S1→S3 as time. Pheno-GS motivates respecting curved, sparse support; it does **not** authorize fabricating graph bridges or forcing continuity.

In [ ]:
from pathlib import Path
import argparse, hashlib, json, os, sys
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr, kendalltau
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(REPO))
from pseudospace.repeated_structure import fit_atlas, map_atlas, select_panel, refine_coordinate, evaluate_prediction, simulate_repeated
from pseudospace.repeated_baselines import baseline_coordinates, anatomy_initialize, curve_agreement, topology_coordinate
from pseudospace.stage_cache import cached_payload
from pseudospace.stats_gam import make_gam_design, gam_internal_knots, dense_gam_sse, single_smooth_slice

parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', default=os.environ.get('PSEUDOSPACE_DATA_ROOT', str(REPO / 'data')))
parser.add_argument('--results-root', default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT', str(REPO / 'results')))
args, _ = parser.parse_known_args()
DATA_ROOT, RESULTS_ROOT = Path(args.data_root), Path(args.results_root)
SOURCE = RESULTS_ROOT / 'mouse_only_v6' / 'all_mouse_tubules_scanpy_dpt.h5ad'
OUT = RESULTS_ROOT / 'novel_repeated_structure_pseudospace'
OUT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_LOGIC_VERSION = '15.1'
SEED = 15
PANELS = (10, 20, 50, 100)
ITERATIONS, RADIUS = 3, .08
N_BOOT = int(os.environ.get('PSEUDOSPACE_15_BOOTSTRAPS', '20'))
SIM_SEEDS = range(int(os.environ.get('PSEUDOSPACE_15_SIM_SEEDS', '3')))
RUN_REAL = SOURCE.exists()
RUN_PANEL_DPT = os.environ.get('PSEUDOSPACE_15_PANEL_DPT', '1') == '1'
# Numerical thresholds are exploratory decisions, not inferential significance.
MIN_GAIN = .05
MARKERS = set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Aqp2 Atp6v0d2 Rhcg Aqp4 Slc14a2 Wnt7b'.split())
ANATOMY_NAMES = np.array(['S1', 'S2', 'S3'])
CODE_FILES = [REPO / 'pseudospace' / f for f in ('repeated_structure.py', 'repeated_baselines.py')]
CODE_DIGEST = hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
plt.rcParams.update({'font.size': 8, 'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'axes.spines.top': False, 'axes.spines.right': False})
print('Current mouse input available:', RUN_REAL)
if not RUN_REAL:
    print('Real-data sections are pending: run reviewed notebook 02 to produce the v6 DPT export. No v5 fallback.')

### Run budget and figure contract

The full real-data run densifies the PT lognorm matrix and evaluates full-gene panels; budget RAM from the printed estimate and expect a long run. All structures are retained. Fold fits are cached; bootstraps repeat fitting and can be reduced for an explicitly labeled smoke run with `PSEUDOSPACE_15_BOOTSTRAPS`. `PSEUDOSPACE_15_SIM_SEEDS` controls synthetic repetitions and `PSEUDOSPACE_15_PANEL_DPT=0` disables optional Scanpy graph baselines. Such reduced runs cannot satisfy the complete benchmark. Save results outside Git. No old notebook is executed or edited here.

### Figure contract

Question: does repeated-specimen prediction supply fine positional information beyond anatomy and conventional geometry? A/B are conceptual, C compares ordering, D is the principal held-out-gene prediction evidence, E tests known synthetic truth, F shows named feature panels, G shows conditional stability. Every quantitative panel exports its CSV source, SVG/PDF and a preview. Points are specimens or simulation seeds where specified; bootstrap spreads are stability ranges, not calibrated spatial confidence intervals. These are exploratory single panels. Publication assembly, collision review and journal-specific checks remain a later step.

In [ ]:
def save_panel(fig, stem, source=None):
    fig.tight_layout()
    fig.savefig(OUT / f'{stem}.svg')
    fig.savefig(OUT / f'{stem}.pdf')
    fig.savefig(OUT / f'{stem}.png', dpi=300)
    if source is not None:
        source.to_csv(OUT / f'{stem}_source.csv', index=False)
    plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 2.6))
ax.axis('off')
ax.text(.03, .85, 'Longitudinal: same instance, linked observations', transform=ax.transAxes)
ax.text(.03, .66, 'Instance A:  time 1 → time 2 → time 3', transform=ax.transAxes)
ax.text(.03, .43, 'Repeated cross-sections: independent instances, no links', transform=ax.transAxes)
ax.text(.03, .23, 'Specimens 1…S:  profiles ○ ○ ○  →  canonical molecular order', transform=ax.transAxes)
save_panel(fig, 'A_problem')
fig, ax = plt.subplots(figsize=(7.2, 2.5))
ax.axis('off')
ax.text(.03, .8, 'Train specimens → select reproducible genes → shared spline atlas', transform=ax.transAxes)
ax.text(.03, .56, 'Leave each training specimen out → refine local molecular order', transform=ax.transAxes)
ax.text(.03, .32, 'New profile → frozen atlas mapping → unseen-gene prediction', transform=ax.transAxes)
ax.text(.03, .08, 'Optional external Geneformer prior → selection only; test added value', transform=ax.transAxes)
save_panel(fig, 'B_method')

## 3 — Current baseline and exact evaluation universe

Read the **reviewed v6 export of 02**, preserving its structures, normalization and annotations. Do not reconstruct different tubule aggregates or reapply count QC. Keep `layers['lognorm']` (normalize-total 10,000 then log1p), `sample`, `condition`, `segment_class`, `coarse_class`, and `total_scanpy_dpt`. Notebook 04 compares different feature universes, so it provides context rather than a benchmark matrix. Notebook 13 is cross-species and supplies methodological context, not this mouse expression input.

The current saved Harmony+DPT is **transductive**: all specimens, evaluation genes and orientation markers helped construct it. A loss computed at those coordinates is a reference diagnostic, not strict unseen-specimen or marker-held-out validation. Anatomy labels were also assigned using molecular markers. Complete marker exclusion below removes direct gene use but cannot remove inherited information in these annotations. A strict sensitivity uses fresh training-only initialization from reconstruction genes; independent histological labels would be needed to eliminate indirect supervision.

In [ ]:
adata = None
X = genes = specimens = anatomy = z_dpt = conditions = structure_ids = None
input_manifest = {'source_workflow': '02_mouse_only_pseudospace', 'required_version': 'mouse_only_v6', 'real_data_available': RUN_REAL}
if RUN_REAL:
    import anndata as ad
    adata = ad.read_h5ad(SOURCE)
    required = {'sample', 'condition', 'segment_class', 'coarse_class', 'total_scanpy_dpt'}
    if not required.issubset(adata.obs) or 'lognorm' not in adata.layers:
        raise ValueError('Current mouse export does not satisfy notebook 02 data contract')
    pt = adata.obs['coarse_class'].astype(str).eq('PT').to_numpy()
    if not pt.any():
        raise ValueError('No PT profiles in current export')
    adata = adata[pt].copy()
    fine = adata.obs['segment_class'].astype(str)
    if not fine.isin(['PT-S1', 'PT-S2', 'PT-S3']).all():
        raise ValueError('Resolve uncertain PT anatomical intervals upstream; do not silently drop structures')
    specimens = adata.obs['sample'].astype(str).to_numpy()
    if set(specimens) != {'Ctrl1A2', 'Ctrl1A4', 'IR2A2', 'IR2A4'}:
        raise ValueError('Unexpected mouse specimen universe')
    conditions = adata.obs['condition'].astype(str).to_numpy()
    anatomy = fine.map({'PT-S1': 0, 'PT-S2': 1, 'PT-S3': 2}).to_numpy()
    z_dpt = adata.obs['total_scanpy_dpt'].to_numpy(float)
    if not np.isfinite(z_dpt).all() or np.ptp(z_dpt) == 0:
        raise ValueError('DPT coordinate is non-finite or constant; repair upstream')
    z_dpt = (z_dpt - z_dpt.min()) / np.ptp(z_dpt)
    genes = adata.var_names.to_numpy(str)
    if len(set(genes)) != len(genes):
        raise ValueError('Gene names must be unique')
    # Retain every structure. Dense RAM estimate is printed before allocation.
    print('Dense expression RAM GiB:', adata.n_obs * adata.n_vars * 8 / 2**30)
    matrix = adata.layers['lognorm']
    X = matrix.toarray().astype(float) if sparse.issparse(matrix) else np.asarray(matrix, float)
    if not np.isfinite(X).all():
        raise ValueError('Non-finite expression values')
    structure_ids = np.array([f'{s}|{o}' for s, o in zip(specimens, adata.obs_names)])
    if len(set(structure_ids)) != len(structure_ids):
        raise ValueError('Structure IDs are not unique')
    input_manifest.update(n_structures=len(X), n_genes=len(genes), structures_sha256=hashlib.sha256('|'.join(structure_ids).encode()).hexdigest(), source_size=SOURCE.stat().st_size, source_mtime_ns=SOURCE.stat().st_mtime_ns)
    display(pd.crosstab(specimens, ANATOMY_NAMES[anatomy]))
    display(pd.crosstab(specimens, conditions))
(OUT / 'input_manifest.json').write_text(json.dumps(input_manifest, indent=2))

## 4 — Does replicate agreement contain information beyond DPT?

Start with the two healthy specimens. Compare per-specimen curves over common support. Split genes before perturbing coordinates; canonical markers and a deterministic 20% evaluation set never enter the reconstruction panel. Modest local changes preserve the **DPT interval range of each coarse segment**, not an arbitrary new coarse ordering. Compare prediction of excluded genes before and after refinement. Positive fitting-gene gains alone are expected under optimization and are not evidence.

This first diagnostic inherits DPT/anatomy information and is deliberately labeled transductive. Section 9 supplies the training-only sensitivity. With two controls, healthy leave-one-out training has only one specimen: replicate-informed training with independent healthy holdout is impossible. Pooling AKI is a stress test of invariance, not additional healthy replication.

In [ ]:
def gene_partition(names):
    marker = np.array([g.casefold() in {m.casefold() for m in MARKERS} for g in names])
    evaluation = np.array([int(hashlib.sha256(g.encode()).hexdigest()[:8], 16) % 5 == 0 for g in names]) & ~marker
    return np.flatnonzero(~marker & ~evaluation), np.flatnonzero(evaluation), np.flatnonzero(marker)

def interval_gauge(z, a):
    # Monotone rescaling within each anatomical interval; report separately from raw DPT.
    out = np.empty(len(z))
    for label in range(3):
        take = a == label
        values = z[take]
        span = np.ptp(values) if len(values) else 0
        out[take] = (label + (.5 if span == 0 else (values - values.min()) / span)) / 3
    return out

def local_dpt_refinement(Y, z, s, a):
    # Refinement helper uses thirds; map back to original per-segment DPT ranges.
    gauged = interval_gauge(z, a)
    new, history = refine_coordinate(Y, gauged, s, a, iterations=ITERATIONS, radius=RADIUS)
    restored = np.empty(len(z))
    for label in range(3):
        take = a == label
        if take.any():
            restored[take] = z[take].min() + (new[take] * 3 - label) * np.ptp(z[take])
    return restored, history

go_table = pd.DataFrame()
reconstruction_genes = evaluation_genes = marker_genes = None
if RUN_REAL:
    reconstruction_genes, evaluation_genes, marker_genes = gene_partition(genes)
    healthy = np.isin(specimens, ['Ctrl1A2', 'Ctrl1A4'])
    panel, _ = select_panel(X[healthy][:, reconstruction_genes], z_dpt[healthy], specimens[healthy], anatomy[healthy], min(50, len(reconstruction_genes)))
    selected = reconstruction_genes[panel]
    z_local, local_history = local_dpt_refinement(X[healthy][:, selected], z_dpt[healthy], specimens[healthy], anatomy[healthy])
    agreement = curve_agreement(X[healthy][:, evaluation_genes], z_dpt[healthy], specimens[healthy], genes[evaluation_genes])
    agreement.to_csv(OUT / 'healthy_dpt_curve_agreement.csv', index=False)
    rows = []
    for holdout in np.unique(specimens[healthy]):
        tr = healthy & (specimens != holdout)
        te = healthy & (specimens == holdout)
        for method, zz in [('DPT reference', z_dpt[healthy]), ('Local refined', z_local)]:
            hs = specimens[healthy]
            scores = evaluate_prediction(X[tr][:, evaluation_genes], zz[hs != holdout], specimens[tr], X[te][:, evaluation_genes], zz[hs == holdout], genes[evaluation_genes])
            rows.append({'specimen': holdout, 'method': method, 'mse': scores['mean_mse']})
    go_table = pd.DataFrame(rows)
    display(go_table)
    go_table.to_csv(OUT / 'go_no_go_transductive.csv', index=False)
    print('Healthy DPT/local rank agreement:', spearmanr(z_dpt[healthy], z_local).statistic)
    print('Diagnostic only: selection and iterative refinement reused both controls; outer validation still required.')

## 5 — Simplest replicate-consistent estimator

Within **training specimens only**, alternate: fit a specimen-balanced P-spline atlas without specimen $s$; map its reconstruction-gene profiles to a finite grid near their current position; update simultaneously. Then fit the final atlas on all training specimens. New structures map onto this frozen atlas without refitting its curves, scaler or gene panel. Finite-grid least squares is standard curve projection, not a new architecture.

The update is a heuristic for a cross-specimen prediction objective, **not** an exact optimizer or a proof of monotone objective descent. After multiple training iterations, other specimens' coordinates carry feedback from prior updates; this leave-one-specimen-out update is not statistical independence within the training set. Only the untouched outer specimen establishes prospective separation. Additive specimen offsets are not fitted to an unseen specimen: raw log-normalized prediction therefore penalizes them. Report within-segment, mean-centered **shape-only** scores separately; those use evaluation responses solely to calculate the metric and never to assign positions. Position/offset confounding remains a failure mode. Hard thirds in this first estimator deliberately exceed the proposed weak anatomy penalty: ablate anatomy during mapping and treat the hard constraint as a prototype ceiling.

Two fixed initializations (PCA within anatomy and its within-interval reflection) diagnose local optima. Selection scores use positive curve agreement and within-segment spline variation, not strongest S1–S3 differential genes. No outer-test metric selects hyperparameters or an initialization. Fixed settings avoid pretending that three training specimens support reliable nested tuning.

In [ ]:
def fit_reconstruction(Y, a, s, k, prior=None, initial=None, reflected=False, specimen_weights=None):
    z0 = anatomy_initialize(Y, ANATOMY_NAMES[a], seed=SEED) if initial is None else initial.copy()
    if reflected:
        z0 = (2 * a + 1) / 3 - z0
    panel, scores = select_panel(Y, z0, s, a, min(k, Y.shape[1]), prior=prior)
    zz, history = refine_coordinate(Y[:, panel], z0, s, a, iterations=ITERATIONS, radius=RADIUS, specimen_weights=specimen_weights)
    atlas = fit_atlas(Y[:, panel], zz, s, specimen_weights=specimen_weights)
    return panel, scores, zz, atlas, history

# Fixed source snapshot and helper code are part of the cache key; validation stays uncached.
def cached_fit(Y, a, s, k, stage, prior=None):
    fingerprint = hashlib.sha256(Y.tobytes() + a.tobytes() + s.astype('U').tobytes()).hexdigest()
    def compute():
        panel, scores, zz, _, _ = fit_reconstruction(Y, a, s, k, prior=prior)
        return {'panel': panel, 'scores': scores, 'z_train': zz}
    payload = cached_payload(stage, compute, root=OUT / 'stage_cache', params={'version': NOTEBOOK_LOGIC_VERSION, 'k': k, 'iterations': ITERATIONS, 'radius': RADIUS, 'seed': SEED, 'prior': None if prior is None else hashlib.sha256(prior.tobytes()).hexdigest()}, inputs={'expression': fingerprint, 'manifest': input_manifest}, code=CODE_DIGEST)
    panel, zz = payload['panel'].astype(int), payload['z_train']
    return panel, payload['scores'], zz, fit_atlas(Y[:, panel], zz, s)

## 6 — Sparse named molecular panels

Evaluate K = 10, 20, 50, 100 and the full reconstruction-gene universe, with selection repeated inside every outer fold. Save named panels and selection frequencies. Compare DPT using exactly the selected panel, PCA1 using that panel, and the replicate atlas using that panel. Ranking does not enforce nonredundancy: correlated-gene block holdout is a required follow-up before a strong claim about independent molecular programs. Deterministic gene holdout here protects direct reuse but does not make correlated genes independent.

## 7 — Geneformer: representation versus prior

[Official model card](https://huggingface.co/ctheodoris/Geneformer/blob/main/README.md) and [tokenizer contract](https://geneformer.readthedocs.io/en/latest/geneformer.tokenizer.html): V2-104M has 104M parameters and was pretrained on approximately 104M **human single-cell** transcriptomes. Input is rank-value encoding from raw, unselected transcriptome counts, adjusted by pretrained gene-expression medians, with Ensembl gene identifiers, V2 token dictionary, CLS/EOS tokens and 4096-token input. Counts metadata must match the official tokenizer. Do not substitute lognorm/HVG values.

Our mouse segmented tubule aggregates sum many Visium HD bins and potentially multiple cell types. They are neither single cells nor validated single-cell-like profiles; ortholog conversion does not resolve this distribution shift. **Direct V2 inference on these profiles is not justified by the documented training contract and is disabled.** Underlying 2µm bins are not automatically single cells either. A defensible route is an independently sourced, suitably matched human PT single-cell dataset, model-supported perturbation/deletion analyses, and a documented one-to-one ortholog transfer. A mouse-trained model is a separate sensitivity, not V2-104M.

A cell embedding→DPT experiment is a **representation baseline**. A perturbation-derived gene prior is a **feature-selection prior**. Contextual token embeddings are not automatically gene importance; virtual perturbation sensitivity is not causal TF regulation.

Optional externally computed priors use a CSV (`gene`, `score`) plus a provenance JSON: model/checkpoint revision, dataset accession, raw-count tokenizer settings, importance procedure, ortholog version and `scope="external"`. Only truly external frozen priors are accepted here. Dataset-derived importance needs fold-specific computation and is deliberately refused by this route. Set `PSEUDOSPACE_15_GF_PRIOR` and `PSEUDOSPACE_15_GF_MANIFEST`; no model downloads or private uploads occur.

TF branch: supply a curated, versioned mouse gene CSV (`gene`) through `PSEUDOSPACE_15_TF_LIST`. Restrict training selection to TFs and use held-out non-TF genes as outputs. This tests predictive TF panels, not regulatory causality. Markers, coordinate features and upstream regulators remain distinct.

In [ ]:
gf_prior = None
tf_indices = None
gf_status = 'Not run: aggregated mouse profiles are outside validated V2 input; no external prior provided.'
prior_path = os.environ.get('PSEUDOSPACE_15_GF_PRIOR')
if prior_path:
    manifest_path = os.environ.get('PSEUDOSPACE_15_GF_MANIFEST')
    if not manifest_path:
        raise ValueError('Geneformer prior requires provenance manifest')
    provenance = json.loads(Path(manifest_path).read_text())
    required_prior = {'scope', 'model_revision', 'dataset_accession', 'tokenizer', 'importance_procedure', 'ortholog_version'}
    if not required_prior.issubset(provenance) or provenance['scope'] != 'external':
        raise ValueError('Only a documented external frozen prior is supported; recompute dataset-dependent priors inside folds')
    prior_frame = pd.read_csv(prior_path)
    if not {'gene', 'score'}.issubset(prior_frame) or prior_frame['gene'].duplicated().any() or not np.isfinite(prior_frame['score']).all():
        raise ValueError('Invalid external gene prior')
    if RUN_REAL:
        gf_prior = prior_frame.set_index('gene')['score'].reindex(genes, fill_value=0).to_numpy(float)
    gf_status = 'External prior supplied; complementary value evaluated by matched-panel held-out prediction.'
tf_path = os.environ.get('PSEUDOSPACE_15_TF_LIST')
if tf_path and RUN_REAL:
    tf_frame = pd.read_csv(tf_path)
    if 'gene' not in tf_frame:
        raise ValueError('TF list must have gene column; retain curated source/version alongside file')
    tf_indices = np.intersect1d(reconstruction_genes, np.flatnonzero(np.isin(genes, tf_frame['gene'])))
print(gf_status)

## 8 — Geometry/anatomy regularization, only as a failure diagnostic

A conventional harmonic coordinate on a kNN graph forbids S1–S3 shortcuts and pins broad S1/S3 endpoints. It necessarily loses within-endpoint resolution, so it is an anatomy stress baseline, not a competing novelty claim. Disconnected components without both endpoint classes return missing coordinates; do not bridge absent anatomy to obtain a prettier curve. Check molecular nearest-neighbor distances at anatomical boundaries against within-region distances before assuming smooth transitions. Geodesic OT is not implemented without evidence that this failure mode matters. An unconstrained frozen-atlas mapping is the minimal anatomy ablation.

## 9 — Cross-validated real-data comparison

Outer folds leave one of four mice out. Selection, standardization, PCA/graph construction, local refinement and atlas training use only the other three. Evaluation responses never map the test profiles. This is an **all-condition invariance stress test**, since AKI modifies positional biology; it cannot validate healthy replication. The strict method starts from reconstruction-gene PCA/anatomy, not saved DPT. Current PCA/Harmony+DPT is retained as a separately marked transductive reference. Training-only selected-panel DPT uses frozen PCA and kNN interpolation because Harmony's joint embedding is not a frozen unseen-specimen transformer; label it accordingly.

Metrics: gene-level raw lognorm MSE, relative improvement over a specimen-balanced training mean, within-specimen/segment shape MSE, coarse ordering and DPT rank agreement. Compare excluded markers only after inference. Report each specimen, not tubule-level significance. Shuffled positions test prediction; shuffled training specimen labels test whether the replicate objective actually uses group structure. Bootstrap specimens/structures/gene sets separately and call resulting ranges conditional stability. Replicate count is too small for population coverage claims.

In [ ]:
baseline_failures = []

def checked_baseline(Xtr, Xte, a_tr, method, context, seed=SEED):
    try:
        return baseline_coordinates(Xtr, Xte, ANATOMY_NAMES[a_tr], method=method, seed=seed)
    except ValueError as exc:
        # A disconnected DPT is a benchmark failure, never an invented graph bridge.
        if method != 'panel_dpt' or 'non-finite training coordinates' not in str(exc):
            raise
        baseline_failures.append({'context': context, 'method': method, 'reason': str(exc)})
        print('Recorded disconnected DPT baseline:', context)
        return None

def score_method(Xtr, ztr, str_, Xte, zte, ate, names, method, specimen, k):
    result = evaluate_prediction(Xtr, ztr, str_, Xte, zte, names)
    frame = pd.DataFrame(result['gene_metrics'])
    frame['method'], frame['specimen'], frame['k'] = method, specimen, k
    predicted = fit_atlas(Xtr, ztr, str_).predict(zte)
    residual = Xte - predicted
    for label in np.unique(ate):
        take = ate == label
        residual[take] -= residual[take].mean(axis=0)
    frame['within_segment_shape_mse'] = np.mean(residual**2, axis=0)
    frame['relative_gain'] = frame['improvement'] / np.maximum(frame['baseline_mse'], 1e-12)
    return frame

metrics, panel_records, positions, boot_records, boot_curve_records = [], [], [], [], []
fold_summary = panels = stability = selected_positions = pd.DataFrame()
if RUN_REAL:
    candidates = [('Replicate atlas', reconstruction_genes, None)]
    if gf_prior is not None:
        candidates.append(('GF prior + replicate atlas', reconstruction_genes, gf_prior[reconstruction_genes]))
    if tf_indices is not None and len(tf_indices) >= 10:
        candidates.append(('TF replicate atlas', tf_indices, None if gf_prior is None else gf_prior[tf_indices]))
    for held in np.unique(specimens):
        tr, te = specimens != held, specimens == held
        a_tr, a_te, s_tr = anatomy[tr], anatomy[te], specimens[tr]
        eval_idx = np.r_[evaluation_genes, marker_genes]
        for candidate, eligible, prior in candidates:
            for k in (*PANELS, len(eligible)):
                if k > len(eligible):
                    continue
                stage = f'fold_{held}_{candidate.replace(" ", "_")}_{k}'
                panel, scores, ztr, atlas = cached_fit(X[tr][:, eligible], a_tr, s_tr, k, stage, prior)
                chosen = eligible[panel]
                zte = map_atlas(X[te][:, chosen], atlas, anatomy=a_te)
                metrics.append(score_method(X[tr][:, eval_idx], ztr, s_tr, X[te][:, eval_idx], zte, a_te, genes[eval_idx], candidate, held, k))
                panel_records.extend({'specimen': held, 'method': candidate, 'k': k, 'gene': genes[g], 'score': float(scores[panel[j]]), 'prior_score': 0. if gf_prior is None else float(gf_prior[g])} for j, g in enumerate(chosen))
                positions.append(pd.DataFrame({'structure': structure_ids[te], 'specimen': held, 'anatomy': a_te, 'method': candidate, 'k': k, 'z': zte, 'dpt_reference': z_dpt[te]}))
                if candidate == 'Replicate atlas' and k == 50:
                    variants = {'Coarse labels only': ((a_tr+.5)/3, (a_te+.5)/3), 'Saved Harmony DPT (transductive)': (z_dpt[tr], z_dpt[te])}
                    for baseline in ['pca1'] + (['panel_dpt'] if RUN_PANEL_DPT else []):
                        baseline_result = checked_baseline(X[tr][:, chosen], X[te][:, chosen], a_tr, baseline, f'real_{held}')
                        if baseline_result is not None:
                            variants[baseline] = baseline_result
                    variants['Atlas without test anatomy'] = (ztr, map_atlas(X[te][:, chosen], atlas))
                    harmonic, graph_diagnostics = topology_coordinate(X[tr][:, chosen], ANATOMY_NAMES[a_tr])
                    (OUT / f'topology_{held}.json').write_text(json.dumps(graph_diagnostics, indent=2))
                    if np.isfinite(harmonic).all() and np.ptp(harmonic) > 0:
                        # Geometry is training-only; frozen nearest-profile mapping for test.
                        from sklearn.preprocessing import StandardScaler
                        from sklearn.neighbors import KNeighborsRegressor
                        graph_scaler = StandardScaler().fit(X[tr][:, chosen])
                        graph_mapper = KNeighborsRegressor(n_neighbors=5, weights='distance').fit(graph_scaler.transform(X[tr][:, chosen]), harmonic)
                        variants['Harmonic anatomy graph'] = (harmonic, graph_mapper.predict(graph_scaler.transform(X[te][:, chosen])))
                    else:
                        print('Harmonic baseline undefined for some training components:', held)
                    # Quantify molecular separation at adjacent anatomical boundaries.
                    from sklearn.neighbors import NearestNeighbors
                    nn = NearestNeighbors(n_neighbors=min(16, np.sum(tr))).fit(X[tr][:, chosen])
                    distances, neighbors = nn.kneighbors(X[tr][:, chosen])
                    boundary = a_tr[:, None] != a_tr[neighbors[:, 1:]]
                    separation = pd.DataFrame({'distance': distances[:, 1:].ravel(), 'cross_anatomy': boundary.ravel()})
                    separation.groupby('cross_anatomy')['distance'].agg(['count', 'median', 'mean']).to_csv(OUT / f'boundary_distances_{held}.csv')
                    reflected_panel, _, reflected_z, reflected_atlas, _ = fit_reconstruction(X[tr][:, eligible], a_tr, s_tr, k, reflected=True)
                    variants['Reflected initialization'] = (reflected_z, map_atlas(X[te][:, eligible[reflected_panel]], reflected_atlas, anatomy=a_te))
                    rng = np.random.default_rng(SEED)
                    variants['Shuffled test positions'] = (ztr, rng.permutation(zte))
                    shuffled_s = rng.permutation(s_tr)
                    spanel, _, sz, sa, _ = fit_reconstruction(X[tr][:, eligible], a_tr, shuffled_s, k)
                    variants['Shuffled training specimens'] = (sz, map_atlas(X[te][:, eligible[spanel]], sa, anatomy=a_te))
                    if gf_prior is not None:
                        gf_panel = eligible[np.argsort(-gf_prior[eligible], kind='stable')[:k]]
                        gf_baseline = checked_baseline(X[tr][:, gf_panel], X[te][:, gf_panel], a_tr, 'panel_dpt', f'GF_{held}')
                        if gf_baseline is not None:
                            variants['GF ranked genes + DPT'] = gf_baseline
                    for method, (zt, zv) in variants.items():
                        metrics.append(score_method(X[tr][:, eval_idx], zt, s_tr, X[te][:, eval_idx], zv, a_te, genes[eval_idx], method, held, k))
                    for boot in range(N_BOOT):
                        for kind in ('specimens', 'structures', 'genes'):
                            rows = np.flatnonzero(tr)
                            boot_genes = eligible
                            multiplicities = None
                            if kind == 'specimens':
                                # Keep original specimen IDs: duplicates are not new independent replicates.
                                labels = rng.choice(np.unique(s_tr), len(np.unique(s_tr)), replace=True)
                                if len(set(labels)) < 2:
                                    continue
                                rows = np.concatenate([np.flatnonzero(specimens == lab) for lab in labels])
                                multiplicities = {lab: int(np.sum(labels == lab)) for lab in set(labels)}
                            elif kind == 'structures':
                                rows = np.concatenate([rng.choice(np.flatnonzero(specimens == lab), np.sum(specimens == lab), replace=True) for lab in np.unique(s_tr)])
                            else:
                                boot_genes = rng.choice(eligible, max(k, int(.8*len(eligible))), replace=False)
                            bp, _, bz, ba, _ = fit_reconstruction(X[rows][:, boot_genes], anatomy[rows], specimens[rows], k, specimen_weights=multiplicities)
                            mapped = map_atlas(X[te][:, boot_genes[bp]], ba, anatomy=a_te)
                            boot_records.append(pd.DataFrame({'structure': structure_ids[te], 'specimen': held, 'kind': kind, 'bootstrap': boot, 'z': mapped}))
                            fixed_genes = evaluation_genes[:6]
                            evaluation_atlas = fit_atlas(X[rows][:, fixed_genes], bz, specimens[rows], specimen_weights=multiplicities)
                            grid = np.linspace(0, 1, 101)
                            curves = evaluation_atlas.predict(grid)
                            for j, g in enumerate(fixed_genes):
                                peak = float(grid[np.argmax(curves[:, j])])
                                transition = float(grid[np.argmax(np.abs(np.gradient(curves[:, j], grid)))])
                                boot_curve_records.extend({'specimen': held, 'kind': kind, 'bootstrap': boot, 'gene': genes[g], 'z': zz, 'expression': yy, 'peak': peak, 'transition': transition} for zz, yy in zip(grid, curves[:, j]))
    metrics = pd.concat(metrics, ignore_index=True)
    metrics['evaluation_role'] = np.where(metrics['gene'].isin(genes[marker_genes]), 'held-out marker (inherited-label caveat)', 'held-out gene')
    panels = pd.DataFrame(panel_records)
    positions = pd.concat(positions, ignore_index=True)
    metrics.to_csv(OUT / 'heldout_gene_metrics.csv', index=False)
    panels.to_csv(OUT / 'selected_gene_panels.csv', index=False)
    positions.to_csv(OUT / 'heldout_positions.csv', index=False)
    fold_summary = metrics.groupby(['specimen', 'method', 'k'], as_index=False)[['mse', 'relative_gain', 'within_segment_shape_mse']].mean()
    fold_summary.to_csv(OUT / 'specimen_metrics.csv', index=False)
    display(fold_summary)
    if boot_records:
        bootstrap = pd.concat(boot_records, ignore_index=True)
        bootstrap.to_csv(OUT / 'conditional_bootstrap_positions.csv', index=False)
        stability = bootstrap.groupby(['structure', 'specimen', 'kind'])['z'].agg(low=lambda x: x.quantile(.025), median='median', high=lambda x: x.quantile(.975), n='count').reset_index()
        stability.to_csv(OUT / 'position_stability.csv', index=False)
        curve_bootstrap = pd.DataFrame(boot_curve_records)
        curve_bootstrap.to_csv(OUT / 'bootstrap_fixed_gene_curves.csv', index=False)
        curve_bootstrap.groupby(['specimen', 'kind', 'gene', 'z'])['expression'].agg(low=lambda x: x.quantile(.025), median='median', high=lambda x: x.quantile(.975)).to_csv(OUT / 'fixed_gene_curve_stability.csv')
        # Fixed adjacent ID pairs give a bounded, reproducible rank stability diagnostic.
        pair_records = []
        for (held, kind), group in bootstrap.groupby(['specimen', 'kind']):
            wide = group.pivot(index='bootstrap', columns='structure', values='z')
            for left, right in zip(wide.columns[:-1], wide.columns[1:]):
                pair_records.append({'specimen': held, 'kind': kind, 'left': left, 'right': right, 'prob_left_before': float((wide[left] < wide[right]).mean()), 'tie_fraction': float((wide[left] == wide[right]).mean())})
        pd.DataFrame(pair_records).to_csv(OUT / 'fixed_pair_rank_stability.csv', index=False)

In [ ]:
if RUN_REAL:
    # Each panel is a single axis. Source records contain all retained structures.
    selected_positions = positions[(positions['method'] == 'Replicate atlas') & (positions['k'] == 50)]
    fig, ax = plt.subplots(figsize=(3.5, 3))
    ax.scatter(selected_positions['dpt_reference'], selected_positions['z'], s=2, alpha=.25, rasterized=True)
    ax.set(xlabel='Saved DPT (transductive)', ylabel='Held-out replicate atlas')
    save_panel(fig, 'C_order_comparison', selected_positions)
    primary = fold_summary[fold_summary['k'] == 50]
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    labels = list(primary['method'].unique())
    for j, method in enumerate(labels):
        subset = primary[primary['method'] == method]
        ax.scatter(subset['relative_gain'], np.full(len(subset), j), s=15, color='#26788e' if method == 'Replicate atlas' else '#8b6f47' if method == 'GF prior + replicate atlas' else '.55')
    ax.axvline(0, color='.7', lw=.6)
    ax.set_yticks(range(len(labels)), [{'pca1': 'PCA1 (selected panel)', 'panel_dpt': 'DPT (selected panel)'}.get(label, label) for label in labels])
    ax.set_xlabel('Held-out expression gain over mean')
    save_panel(fig, 'D_generalization', primary)
    # Descriptive heatmap uses only reconstruction genes; it is not validation.
    one = panels[(panels['method'] == 'Replicate atlas') & (panels['k'] == 20)]
    held = sorted(one['specimen'].unique())[0]
    named = one[one['specimen'] == held]['gene'].to_numpy()
    cols = pd.Index(genes).get_indexer(named)
    rows = np.flatnonzero(specimens == held)
    zz = positions[(positions['specimen'] == held) & (positions['method'] == 'Replicate atlas') & (positions['k'] == 20)]['z'].to_numpy()
    order = np.argsort(zz)
    values = X[rows][:, cols][order]
    values = (values - values.mean(axis=0)) / np.maximum(values.std(axis=0), 1e-8)
    fig, ax = plt.subplots(figsize=(3.5, 4.2))
    ax.imshow(values.T, aspect='auto', cmap='viridis', interpolation='nearest', vmin=-2, vmax=2)
    ax.set_yticks(range(len(named)), named)
    ax.set(xlabel='Held-out profiles ordered by atlas', ylabel='Selected real genes (gene z-score)')
    heatmap_source = pd.DataFrame(values, columns=named)
    heatmap_source.insert(0, 'z', zz[order])
    heatmap_source.insert(0, 'structure', structure_ids[rows][order])
    save_panel(fig, 'F_sparse_panel', heatmap_source)
    if boot_records:
        view = stability[stability['kind'] == 'structures'].sort_values('high').copy()
        view['width'] = view['high'] - view['low']
        fig, ax = plt.subplots(figsize=(3.5, 3))
        ax.scatter(view['median'], view['width'], s=2, rasterized=True)
        ax.set(xlabel='Conditional median coordinate', ylabel='Bootstrap 95% stability width')
        save_panel(fig, 'G_position_stability', view)
    nuisance_rows = []
    for held in np.unique(specimens):
        take = specimens == held
        zp = selected_positions[selected_positions['specimen'] == held]['z'].to_numpy()
        for covariate in ['n_spots', 'total_counts', 'area_um2']:
            if covariate in adata.obs:
                nuisance_rows.append({'specimen': held, 'covariate': covariate, 'spearman': spearmanr(zp, adata.obs.loc[take, covariate]).statistic})
    pd.DataFrame(nuisance_rows).to_csv(OUT / 'nuisance_associations.csv', index=False)
    # Coarse consistency and within-region reproducibility are kept distinct.
    pd.DataFrame([{'specimen': held, 'coarse_spearman': spearmanr(selected_positions[selected_positions['specimen']==held]['z'], anatomy[specimens==held]).statistic} for held in np.unique(specimens)]).to_csv(OUT / 'coarse_consistency.csv', index=False)
    for region in range(3):
        take = anatomy == region
        if len(np.unique(specimens[take])) > 1:
            aligned = selected_positions.set_index('structure').reindex(structure_ids[take])['z'].to_numpy()
            curve_agreement(X[take][:, evaluation_genes], aligned, specimens[take], genes[evaluation_genes]).to_csv(OUT / f'within_S{region+1}_heldout_curves.csv', index=False)

## 10 — Synthetic ground truth and expected failures

Simulate independent profiles with uneven specimen-specific density, monotonic/peak/sigmoid programs, subtle regulator-like programs, correlated modules, null genes, specimen offsets/scaling and strong position-independent nuisance. Missing regions, sharp transitions, specimen-specific warp, reflection-symmetric curves and absent shared signal are explicit failures/stress cases. These are transformed-expression simulations, not a calibrated Visium count generator.

True z is used **only for evaluation**. Inferred positions initialize from reconstruction genes plus coarse labels. Compare full-feature PCA1, selected-panel PCA1/DPT, coarse labels and replicate atlas. A full-feature DPT baseline is also run when enabled. Anatomy supplies considerable information: report within-segment Kendall ordering as well as gross ordering. Compare signed correlations to detect orientation failures. Isotonic alignment is fitted on training truth solely for synthetic metric calibration, then scored on the held-out specimen; it cannot improve the estimator itself.

The symmetric case tests fine-order ambiguity despite anatomy; the null case should retain only imposed coarse order. A method that wins only because anatomy was provided, or loses to the identical-panel baseline, has not demonstrated the proposed contribution.

In [ ]:
from sklearn.isotonic import IsotonicRegression

def truth_metrics(ztrue_train, zfit_train, ztrue_test, zfit_test, atest):
    tau = kendalltau(ztrue_test, zfit_test).statistic
    calibrator = IsotonicRegression(out_of_bounds='clip').fit(zfit_train, ztrue_train)
    aligned = calibrator.predict(zfit_test)
    within = [kendalltau(ztrue_test[atest == a], zfit_test[atest == a]).statistic for a in range(3) if np.sum(atest == a) >= 5]
    # Explicit discordance fraction; ties are reported separately rather than counted as successes.
    dz = ztrue_test[:, None] - ztrue_test[None, :]
    dh = zfit_test[:, None] - zfit_test[None, :]
    upper = np.triu(np.ones(dz.shape, dtype=bool), 1)
    return {'spearman': spearmanr(ztrue_test, zfit_test).statistic, 'kendall': tau, 'inversion_rate': np.mean((dz[upper]*dh[upper]) < 0), 'tie_rate': np.mean(dh[upper] == 0), 'aligned_mae': np.mean(np.abs(aligned-ztrue_test)), 'within_segment_kendall': np.nanmean(within) if np.isfinite(within).any() else np.nan}

sim_rows, sim_positions = [], []
for scenario in ['smooth', 'missing_region', 'sharp', 'symmetric', 'no_shared_signal', 'specimen_warp']:
    for strength in [0., 2., 6.]:
        for seed in SIM_SEEDS:
            SX, truth, ss, aa = simulate_repeated(seed, n_specimens=6, n_per_specimen=80, n_genes=120, nuisance=strength, scenario=scenario)
            # Last specimen is untouched until mapping. Alternate seeds vary the complete simulation.
            train = ss != np.unique(ss)[-1]
            test = ~train
            infer = np.arange(0, SX.shape[1], 2)
            ev = np.arange(1, SX.shape[1], 2)
            panel, _, ztr, atlas, _ = fit_reconstruction(SX[train][:, infer], aa[train], ss[train], 20)
            chosen = infer[panel]
            zte = map_atlas(SX[test][:, chosen], atlas, anatomy=aa[test])
            variants = {'Replicate atlas': (ztr, zte), 'Coarse labels': ((aa[train]+.5)/3, (aa[test]+.5)/3)}
            for label, cols in [('Full-feature PCA1', infer), ('Same-panel PCA1', chosen)]:
                variants[label] = baseline_coordinates(SX[train][:, cols], SX[test][:, cols], ANATOMY_NAMES[aa[train]], method='pca1', seed=seed)
            if RUN_PANEL_DPT:
                for label, cols in [('Full-feature DPT', infer), ('Same-panel DPT', chosen)]:
                    dpt_result = checked_baseline(SX[train][:, cols], SX[test][:, cols], aa[train], 'panel_dpt', f'{scenario}_{strength}_{seed}_{label}', seed=seed)
                    if dpt_result is not None:
                        variants[label] = dpt_result
                    else:
                        sim_rows.append({'scenario': scenario, 'nuisance': strength, 'seed': seed, 'method': label, 'status': 'disconnected graph', 'heldout_mse': np.nan, 'spearman': np.nan, 'kendall': np.nan, 'inversion_rate': np.nan, 'tie_rate': np.nan, 'aligned_mae': np.nan, 'within_segment_kendall': np.nan})
            variants['Shuffled positions'] = (ztr, np.random.default_rng(seed).permutation(zte))
            for method, (fit_train, fit_test) in variants.items():
                scores = truth_metrics(truth[train], fit_train, truth[test], fit_test, aa[test])
                pred = evaluate_prediction(SX[train][:, ev], fit_train, ss[train], SX[test][:, ev], fit_test)
                sim_rows.append({'scenario': scenario, 'nuisance': strength, 'seed': seed, 'method': method, 'heldout_mse': pred['mean_mse'], 'status': 'evaluated', **scores})
                sim_positions.extend({'scenario': scenario, 'nuisance': strength, 'seed': seed, 'method': method, 'truth': t, 'inferred': v} for t,v in zip(truth[test], fit_test))
simulation = pd.DataFrame(sim_rows)
simulation.to_csv(OUT / 'synthetic_metrics.csv', index=False)
pd.DataFrame(baseline_failures, columns=['context', 'method', 'reason']).to_csv(OUT / 'baseline_failures.csv', index=False)
pd.DataFrame(sim_positions).to_csv(OUT / 'synthetic_positions.csv', index=False)
display(simulation.groupby(['scenario', 'nuisance', 'method'])[['spearman', 'within_segment_kendall', 'aligned_mae', 'heldout_mse']].mean())
fig, ax = plt.subplots(figsize=(3.5, 3))
for method, color in [('Full-feature PCA1', '.6'), ('Replicate atlas', '#26788e'), ('Same-panel PCA1', '#8b6f47')]:
    subset = simulation[(simulation['scenario'] == 'smooth') & (simulation['method'] == method)]
    ax.scatter(subset['nuisance'], subset['within_segment_kendall'], label=method, s=18, color=color)
ax.set(xlabel='Nuisance strength', ylabel='Within-segment Kendall ordering')
ax.legend(loc='upper left', bbox_to_anchor=(0, 1.35), fontsize=7)
save_panel(fig, 'E_synthetic_nuisance', simulation)
truth_source = pd.DataFrame(sim_positions)
truth_view = truth_source[(truth_source['scenario'] == 'smooth') & (truth_source['nuisance'] == 6) & (truth_source['seed'] == 0)]
fig, ax = plt.subplots(figsize=(3.5, 3))
for method, color in [('Full-feature DPT', '.6'), ('Replicate atlas', '#26788e')]:
    subset = truth_view[truth_view['method'] == method]
    ax.scatter(subset['truth'], subset['inferred'], label=method, color=color, s=7, alpha=.65)
ax.set(xlabel='Known synthetic position', ylabel='Inferred coordinate (chosen gauge)')
ax.legend(loc='upper left', bbox_to_anchor=(0, 1.3), fontsize=7)
save_panel(fig, 'E_truth_comparison', truth_view)
print('No superiority claim: inspect all scenarios and same-panel controls, including failed orientation.')

### Identifiability experiments and limits

Orientation is fixed by broad anatomy for all training-only baselines; it is not counted as inferred biological direction. Within-interval reflected initialization in Section 9 probes local ambiguity. Symmetry can be broken by coarse labels across regions yet remain unresolved within a region. If programs do not distinguish positions, no molecular estimator can recover them. Missing overlap prevents curve comparison; disconnected geometry must return uncertainty/missing coordinates. Specimen-specific biological warping may make one canonical axis inappropriate. Positional uncertainty ranges are conditional on chosen anatomy, initialization and coordinate gauge; they cannot establish physical calibration.

## 11 — Biological utility after validation

Only advance after excluded-gene prediction improves beyond labels and same-panel DPT in each fold, including within-segment shape, and synthetic within-region truth supports the principle. Even then, healthy validation remains unestablished with two controls. The following fixed gene-first GAM summary is descriptive: choose no genes from test performance and do not rank by p-values. Pathways require an independent validation set and versioned annotation file; cross-species alignment requires new folds with the single human donor kept intact. Those extensions are deferred until the go/no-go survives. More discoveries are not evidence of better reconstruction.

In [ ]:
advance = False
sim_comparison = simulation[simulation['scenario'].eq('smooth') & simulation['nuisance'].ge(2)].groupby(['seed', 'method'])['within_segment_kendall'].mean().unstack()
comparison_methods = [m for m in ['Full-feature PCA1', 'Same-panel PCA1', 'Full-feature DPT', 'Same-panel DPT'] if m in sim_comparison]
sim_gain = sim_comparison[comparison_methods].rsub(sim_comparison['Replicate atlas'], axis=0)
synthetic_support = bool(len(sim_gain) >= 3 and RUN_PANEL_DPT and len(comparison_methods) == 4 and (sim_gain > MIN_GAIN).all().all())
synthetic_verdict = 'Consistent synthetic fine-order advantage at the fixed threshold.' if synthetic_support else 'No consistent synthetic fine-order advantage over the tested conventional baselines at the fixed threshold.'
print(synthetic_verdict)
sim_gain.to_csv(OUT / 'synthetic_paired_ordering_gains.csv')
if RUN_REAL:
    primary_metrics = fold_summary[fold_summary['k'] == 50].pivot(index='specimen', columns='method', values='mse')
    required_methods = ['Replicate atlas', 'Coarse labels only', 'pca1', 'panel_dpt']
    if all(m in primary_metrics for m in required_methods):
        gains = 1 - primary_metrics['Replicate atlas'].to_numpy()[:, None] / primary_metrics[['Coarse labels only', 'pca1', 'panel_dpt']].to_numpy()
        shape_metrics = fold_summary[fold_summary['k'] == 50].pivot(index='specimen', columns='method', values='within_segment_shape_mse')
        shape_gain = 1 - shape_metrics['Replicate atlas'] / shape_metrics['Coarse labels only']
        advance = bool((gains > MIN_GAIN).all() and (shape_gain > MIN_GAIN).all() and synthetic_support)
    print('Exploratory real-data threshold passed:', advance)
    if advance:
        # Fixed evaluation genes only; no selection from significance or curve appearance.
        examples = genes[evaluation_genes][:6]
        gam_rows = []
        for held in np.unique(specimens):
            take = specimens == held
            zz = selected_positions[selected_positions['specimen'] == held]['z'].to_numpy()
            design = make_gam_design(zz, gam_internal_knots(zz))
            yy = X[take][:, pd.Index(genes).get_indexer(examples)]
            sse, beta, edf = dense_gam_sse(yy, design, single_smooth_slice(design))
            gam_rows.extend({'specimen': held, 'gene': g, 'sse': float(error), 'edf': float(edf)} for g,error in zip(examples,sse))
        pd.DataFrame(gam_rows).to_csv(OUT / 'post_gate_fixed_gene_gam.csv', index=False)
else:
    print('Biological utility not evaluated: current mouse v6 input is unavailable.')

## 12 — Critical methodological verdict

The notebook must distinguish runnable implementation from observed evidence. The following answers are written from available artifacts; no missing real result is inferred from simulation. Novelty remains unresolved and is not asserted. Retain conventional DPT unless independent validation supports a better estimator. Applicability to crypt–villus, airway/gland, ducts, vascular segments and follicles is hypothetical and requires shared programs, comparable measurement units and topology-appropriate models; branching structures need more than a scalar coordinate.

In [ ]:
answers = [
    ('1. Inference problem', 'Recover a shared molecular order from unordered independent profiles of repeated structures, without longitudinal correspondence or known fine location.'),
    ('2. Estimator/objective', 'Heuristic specimen-cross-fitted P-spline curve fitting and local projection, with a frozen atlas for new profiles; not an exact optimizer of a new identifiable objective.'),
    ('3. Standard components', 'Spline/GAM regression, cross-fitting, feature ranks, PCA/DPT, kNN mapping, anatomy constraints and atlas projection are established.'),
    ('4. Improvement over DPT', 'Pending current v6 data.' if not RUN_REAL else 'See go_no_go_transductive.csv and specimen_metrics.csv; distinguish local diagnostic from strict training-only sensitivity.'),
    ('5. Held-out specimens', 'Healthy replication is unestablished: only two healthy mice. Four-mouse LOSO is an AKI invariance stress test, even if its threshold passes.'),
    ('6. Unseen genes', 'Direct exclusion is implemented; marker/anatomy inheritance and correlated-gene leakage remain. Real results pending.' if not RUN_REAL else 'Inspect excluded-gene/marker MSE and within-segment shape in heldout_gene_metrics.csv; annotations remain indirect supervision.'),
    ('7. Geneformer', gf_status + ' Representation inference is deferred pending valid single-cell inputs; no benefit assumed.'),
    ('8. Sparse genes', 'Fold-selected 10/20/50/100 panels are implemented; real retained performance is pending.' if not RUN_REAL else 'Compare panels and matched-panel DPT/PCA before selecting a compact panel; outer test folds do not choose K.'),
    ('9. Synthetic truth', synthetic_verdict + ' See synthetic_metrics.csv and paired ordering gains; simulation cannot establish real-data superiority.'),
    ('10. Assumptions', 'Shared injective positional programs, sufficient support overlap, comparable measurement, limited nuisance/warping and an orientation/gauge convention. Agreement alone does not identify metric distance.'),
    ('11. Beyond kidney', 'Plausible hypothesis, untested. Branching or nonrecurring programs invalidate this scalar shared-atlas model.'),
    ('12. Paper verdict', 'No novelty claim. Keep DPT as the established coordinate until independent healthy-specimen evidence and a deeper precedence audit support a methodological contribution.')
]
verdict = pd.DataFrame(answers, columns=['question', 'assessment'])
verdict.to_csv(OUT / 'methodological_verdict.csv', index=False)
display(verdict)
(OUT / 'run_configuration.json').write_text(json.dumps({'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'panels': PANELS, 'iterations': ITERATIONS, 'radius': RADIUS, 'bootstrap_count': N_BOOT, 'simulation_seeds': list(SIM_SEEDS), 'code_digest': CODE_DIGEST, 'real_data_available': RUN_REAL, 'panel_dpt': RUN_PANEL_DPT, 'advance_threshold': MIN_GAIN, 'advance': advance, 'synthetic_support': synthetic_support}, indent=2))